In [4]:
import os, glob, zipfile, urllib.request
import pandas as pd

RAW = "/content/EduPredict/data/raw"
PROC = "/content/EduPredict/data/processed"
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

URL = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"

def find_csv():
    hits = glob.glob("/content/**/data.csv", recursive=True)
    return hits[0] if hits else None

path = find_csv()
if path is None:
    zip_path = f"{RAW}/dropout.zip"
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r, open(zip_path, "wb") as f:
        f.write(r.read())
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(f"{RAW}/dropout")
    path = find_csv()
print("Using:", path)

df = pd.read_csv(path, sep=";")
df.columns = df.columns.str.strip().str.replace("Nacionality", "Nationality", regex=False)
df = df.drop_duplicates().reset_index(drop=True)

categorical = ["Marital status", "Application mode", "Application order", "Course",
               "Previous qualification", "Nationality", "Mother's qualification",
               "Father's qualification", "Mother's occupation", "Father's occupation"]
binary = ["Daytime/evening attendance", "Displaced", "Educational special needs",
          "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "International"]
for c in binary:
    assert set(df[c].unique()) <= {0, 1}, f"{c} has values other than 0/1"
for c in categorical + binary:
    df[c] = df[c].astype("int64")

df["Target"] = df["Target"].astype(str).str.strip().str.capitalize()
assert set(df["Target"].unique()) == {"Dropout", "Enrolled", "Graduate"}
assert df.isnull().sum().sum() == 0

df.to_csv(f"{PROC}/dropout_clean.csv", index=False)
print(pd.read_csv(f"{PROC}/dropout_clean.csv").shape)
print(df["Target"].value_counts().to_dict())

Using: /content/EduPredict/data/raw/dropout/data.csv
(4424, 37)
{'Graduate': 2209, 'Dropout': 1421, 'Enrolled': 794}


In [5]:
import os, json, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, roc_auc_score

warnings.filterwarnings("ignore")

PROC = "/content/EduPredict/data/processed"
MODELS = "/content/EduPredict/models"
os.makedirs(MODELS, exist_ok=True)

if not os.path.exists(f"{PROC}/dropout_clean.csv"):
    raise FileNotFoundError("dropout_clean.csv missing. Colab was reset: re-run Phase 3 (Cells 1-8) first.")

df = pd.read_csv(f"{PROC}/dropout_clean.csv")
df["Dropout_flag"] = (df["Target"] == "Dropout").astype(int)

print(df["Dropout_flag"].value_counts().to_dict(), f"-> {df['Dropout_flag'].mean() * 100:.1f}% dropout")

{0: 3003, 1: 1421} -> 32.1% dropout


In [6]:
# Check near-constant columns (one value covers >95% of students)
candidates = df.drop(columns=["Target", "Dropout_flag"])
near_constant = [c for c in candidates.columns if candidates[c].value_counts(normalize=True).iloc[0] > 0.95]
print("Near-constant:", near_constant)

DROPPED = ["Nationality", "International", "Educational special needs",
           "Mother's qualification", "Father's qualification",
           "Mother's occupation", "Father's occupation",
           "Unemployment rate", "Inflation rate", "GDP"]

ENROLLMENT = ["Marital status", "Application mode", "Application order", "Course",
              "Daytime/evening attendance", "Previous qualification",
              "Previous qualification (grade)", "Admission grade", "Displaced",
              "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder",
              "Age at enrollment"]

SEM1 = [f"Curricular units 1st sem ({s})" for s in
        ["credited", "enrolled", "evaluations", "approved", "grade", "without evaluations"]]
SEM2 = [c.replace("1st", "2nd") for c in SEM1]

FEATURE_SETS = {
    "enrollment": ENROLLMENT,
    "sem1": ENROLLMENT + SEM1,
    "full": ENROLLMENT + SEM1 + SEM2,
}

# Safety check: every column is either used or deliberately dropped
all_features = set(df.columns) - {"Target", "Dropout_flag"}
assert set(FEATURE_SETS["full"]) | set(DROPPED) == all_features, "Some column is unaccounted for!"
assert not set(FEATURE_SETS["full"]) & set(DROPPED)
assert "Target" not in FEATURE_SETS["full"] and "Dropout_flag" not in FEATURE_SETS["full"]  # no target leakage

print({k: len(v) for k, v in FEATURE_SETS.items()})


Near-constant: ['Nationality', 'Educational special needs', 'International']
{'enrollment': 14, 'sem1': 20, 'full': 26}


In [7]:
CATEGORICAL = ["Marital status", "Application mode", "Application order", "Course", "Previous qualification"]
BINARY = ["Daytime/evening attendance", "Displaced", "Debtor",
          "Tuition fees up to date", "Gender", "Scholarship holder"]
NUMERIC = [c for c in FEATURE_SETS["full"] if c not in CATEGORICAL + BINARY]

print(len(NUMERIC), "numeric features")
print(NUMERIC)

15 numeric features
['Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)']


In [8]:
def build_preprocessor(feature_list):
    cat = [c for c in feature_list if c in CATEGORICAL]
    binary = [c for c in feature_list if c in BINARY]
    num = [c for c in feature_list if c in NUMERIC]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num),                       # scale numbers
            ("bin", "passthrough", binary),                       # keep 0/1 as is
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=20), cat),        # codes -> one-hot columns
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

In [9]:
X = df[FEATURE_SETS["full"]]
y = df["Dropout_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print("Dropout share  train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

(3539, 26) (885, 26)
Dropout share  train: 0.321 | test: 0.321


In [10]:
sem1_cols = FEATURE_SETS["sem1"]
prep = build_preprocessor(sem1_cols).fit(X_train[sem1_cols])     # fit on TRAIN only

scaler = prep.named_transformers_["num"]
print("Scaler learned from train:", dict(zip(scaler.feature_names_in_[:3], scaler.mean_[:3].round(2))))
print("Admission grade mean (train):", round(X_train["Admission grade"].mean(), 2),
      "| (all data, NOT used):", round(X["Admission grade"].mean(), 2))

for name, cols in FEATURE_SETS.items():
    p = build_preprocessor(cols).fit(X_train[cols])
    print(f"{name:10s}: {len(cols)} features -> {len(p.get_feature_names_out())} model columns")

Scaler learned from train: {'Previous qualification (grade)': np.float64(132.68), 'Admission grade': np.float64(126.99), 'Age at enrollment': np.float64(23.19)}
Admission grade mean (train): 126.99 | (all data, NOT used): 126.98
enrollment: 14 features -> 60 model columns
sem1      : 20 features -> 66 model columns
full      : 26 features -> 72 model columns


In [11]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
print("Train non-dropout:", neg, "| dropout:", pos, "| ratio:", round(neg / pos, 2))
print("Balanced class weights -> non-dropout:", round(len(y_train) / (2 * neg), 2),
      "| dropout:", round(len(y_train) / (2 * pos), 2))

Train non-dropout: 2402 | dropout: 1137 | ratio: 2.11
Balanced class weights -> non-dropout: 0.74 | dropout: 1.56


In [12]:
for name, cols in FEATURE_SETS.items():
    model = Pipeline([
        ("prep", build_preprocessor(cols)),
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced")),
    ]).fit(X_train[cols], y_train)
    pred = model.predict(X_test[cols])
    prob = model.predict_proba(X_test[cols])[:, 1]
    print(f"{name:10s} dropout recall: {recall_score(y_test, pred):.3f} | ROC-AUC: {roc_auc_score(y_test, prob):.3f}")

enrollment dropout recall: 0.736 | ROC-AUC: 0.836
sem1       dropout recall: 0.810 | ROC-AUC: 0.912
full       dropout recall: 0.831 | ROC-AUC: 0.933


In [13]:
config = {
    "target": "Dropout_flag (1 = Dropout, 0 = Graduate or Enrolled)",
    "feature_sets": FEATURE_SETS,
    "categorical": CATEGORICAL,
    "binary": BINARY,
    "numeric": NUMERIC,
    "dropped_features": DROPPED,
    "split": {"test_size": 0.2, "random_state": 42, "stratify": True},
    "primary_feature_set": "sem1",
}
with open(f"{MODELS}/feature_config.json", "w") as f:
    json.dump(config, f, indent=2)

X_train.assign(Dropout_flag=y_train).to_csv(f"{PROC}/train.csv", index=False)
X_test.assign(Dropout_flag=y_test).to_csv(f"{PROC}/test.csv", index=False)

print(pd.read_csv(f"{PROC}/train.csv").shape, pd.read_csv(f"{PROC}/test.csv").shape)

(3539, 27) (885, 27)


In [14]:
import shutil
from google.colab import files

os.makedirs("/content/phase5_out/models", exist_ok=True)
os.makedirs("/content/phase5_out/data_processed", exist_ok=True)
shutil.copy(f"{MODELS}/feature_config.json", "/content/phase5_out/models/")
shutil.copy(f"{PROC}/train.csv", "/content/phase5_out/data_processed/")
shutil.copy(f"{PROC}/test.csv", "/content/phase5_out/data_processed/")
shutil.make_archive("/content/phase5_out", "zip", "/content/phase5_out")
files.download("/content/phase5_out.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>